In [1]:
FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'
#FILE_PATH = 'D://bitmex_data_1m.csv'

In [ ]:
test_df = pd.read_csv(FILE_PATH, delimiter=',')
test_df

In [19]:
import numpy as np

In [87]:
from abc import ABC, abstractmethod
from common import *
import plotly

# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str = None):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        
        if(self.from_date):
            df = df.loc[df.timestamp >= self.from_date]  #일단 GMT 니까 1분 뒤로 조정할 걸 생각하고 +1분부터 가져오면 된다.
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        data['ma_20']=self.__sma(data,20)
        data['ma_60']=self.__sma(data,60)
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __sma(self, data, period=20):
        return data['close'].rolling(window=period, min_periods=1).mean()
    
# 구체적인 데이터 처리 전략: RSI 계산
class RSIProcessing(DataProcessingStrategy):
    def process_data(self, data):
        data['RSI'] = self.__rsi(data)
        print("RSI를 계산했습니다.")
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __rsi(self, data, period=14):
    
        import numpy as np
        import pandas as pd
        delta = data['close'].diff(1)  # 종가의 변화량 계산
        gain = np.where(delta > 0, delta, 0)  # 상승분
        loss = np.where(delta < 0, -delta, 0)  # 하락분
    
        avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
        avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()
        
        rs = avg_gain / (avg_loss + 1e-10)  # 0으로 나누는 오류 방지
        rsi = 100 - (100 / (1 + rs))
        
        rsi.index = data.index
        
        return rsi

#고점을 찾는 처리 전략
class HighPointScoringProcessing(DataProcessingStrategy):
    '''
    메인 df에 'high_score' 라는 컬럼을 추가하고, 외부 변수의 리스트로 들어온 
    밴드 값을 shift 하면서 그중에 가장 큰 가격에 +1 스코어를 한다.
    '''
    
    
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
    
    def process_data(self, data):
        data = self.__get_high_score_by_list(data,self.bandwith_list)
        return data
    
    def __get_high_score_by_list(self, origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_high_score_with_bandwidth(origin_df, 'high', i)
                else:
                    return_df = self.__get_high_score_with_bandwidth(return_df, 'high', i, reset=False)
    
        return return_df
    
    
        
    def __get_high_score_with_bandwidth(self, target_df, column_name, bandwidth, reset=True):
        '''
        df를 제공하면서 밴드 값을 같이 제공하면 이를 반복문으로 돌아가면서 score 를 쌓는 함수
        '''
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['high_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            max_index = target_df.iloc[i:i+bandwidth][column_name].idxmax()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
        #
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
        #
            #print(f"체크값 :{target_df.loc[max_index]['high_score']+1}")
            #
            target_df.loc[max_index,'high_score'] = target_df.loc[max_index]['high_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
        
        print(f"수행한 숫자:{end_index}")
        print(f"가장 높은 점수:{target_df.loc[target_df['high_score'].idxmax()]['high_score']}")
        
        return target_df

#저점을 찾는 처리 전략
class LowPointScoringProcessing(DataProcessingStrategy):
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
        
    def process_data(self, data):
        data = self.__get_low_score_by_list(data, self.bandwith_list)
        return data
    
    def __get_low_score_by_list(self,origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_low_score_with_bandwidth(origin_df, 'low', i)
                else:
                    return_df = self.__get_low_score_with_bandwidth(return_df, 'low', i, reset=False)
        
        return return_df
    
    
    def __get_low_score_with_bandwidth(self,target_df, column_name, bandwidth, reset=True):
    
    
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['low_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            min_index = target_df.iloc[i:i+bandwidth][column_name].idxmin()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[min_index,'low_score'] = target_df.loc[min_index]['low_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            

        return target_df

# 고점만을 필터팅하는 처리 전략
class GetHighPoints(DataProcessingStrategy):
    def __init__(self, data, target_column, threshold_ratio):
        self.data = data
        self.target_column = target_column
        self.threshold_ratio = threshold_ratio
        
    def process_data(self, data):
        #일단 0점인건 전부 제외
        not_high_score_zero_data =data.loc[data['high_score']!=0]
        final_high_point=self.__apply_threshold_with_normalizing_for_high_value(self.data,self.target_column,self.threshold_ratio)
        
        return final_high_point
        
    #df와 타겟 컬럼명을 받아서 해당 %이상의 값만 필터링하는 함수
    def __apply_threshold_with_normalizing_for_high_value(self, target_df, target_column, threshold):
    
        #min-max 정규화
        target_df['normalized_value_high'] = (target_df[target_column] - target_df[target_column].min()) / (target_df[target_column].max() - target_df[target_column].min())
        
        threshold = target_df['normalized_value_high'].quantile(threshold)
        print(f"정형화된 기준값은 : {threshold}")
        
        #상위 5%에 포함되는 애들을 별도로 선언
    
        df_calculated = target_df[target_df['normalized_value_high'] >= threshold]
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
        #그래프를 그리기 위한 수단을 벌써 넣으면 안됨
        df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'high_for_graph'] = df_calculated['high']
        
        import numpy as np
        df_calculated.loc[df_calculated['normalized_value_high'] < threshold, target_column] = np.nan
        
        return df_calculated
    
    def __merge_high_score_df(self, df_origin, base_score):
        df_calculated=df_origin.loc[df_origin['high_score'] > base_score]
        df_calculated.loc[df_calculated['high_score'] > base_score, 'origin_score'] = df_calculated['high_score']
        df_calculated.loc[df_calculated['high_score'] > base_score, 'high_score'] = df_calculated['high']
    
        import numpy as np
        df_calculated.loc[df_calculated['high_score'] <= base_score, 'high_score'] = np.nan
        
        return df_calculated
    

# 데이터 시각화 전략 인터페이스
class VisualizationStrategy(ABC):
    @abstractmethod
    def visualize(self, data):
        pass

# 구체적인 시각화 전략: 라인 그래프
class LineGraphVisualization(VisualizationStrategy):
    def visualize(self, data):
        print("데이터를 그래프로 시각화합니다.")
        pass

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def add_sub_indicator(self,indicator_instance_list):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.'''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
            
    def get_key_points(self, instance : DataProcessingStrategy):
        '''고점을 찾거나 다이버전스를 찾는등의 주요 포인트를 찾을 때 활용'''
        return instance.process_data(self.data) #이미 세팅된 메인데이터를 활용한다.
            
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [4]:
#bitmex = Bitmex(, MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#bitmex.load()

In [79]:
bitmex = Bitmex()
bitmex.set_loader(BitmexCSVDataLoader(15,'2022-12-31 15:01:00'))
bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
입력받은 15분봉으로 2022-12-31 15:01:00 부터 표현합니다.
CSV 데이터 로드 완료.
전처리 완료


In [80]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int
0,2023-01-01 00:00:00,16583.5,16583.5,16608.0,16600.0,1.672499e+09
10,2023-01-01 00:15:00,16591.5,16583.5,16608.0,16590.5,1.672500e+09
27,2023-01-01 00:45:00,16590.5,16578.0,16590.5,16580.0,1.672502e+09
47,2023-01-01 01:15:00,16583.0,16573.5,16588.5,16588.5,1.672503e+09
66,2023-01-01 01:45:00,16587.5,16585.5,16596.5,16592.5,1.672505e+09
...,...,...,...,...,...,...
174825,2023-05-08 11:00:00,28308.0,28207.0,28374.0,28362.0,1.683511e+09
174840,2023-05-08 11:15:00,28362.0,28293.0,28374.0,28329.5,1.683512e+09
174855,2023-05-08 11:30:00,28329.5,28260.0,28330.5,28330.5,1.683513e+09
174870,2023-05-08 11:45:00,28330.5,28315.5,28395.0,28376.0,1.683514e+09


In [81]:
indicator_list = [MovingAverageProcessing(), RSIProcessing()]
bitmex.add_sub_indicator(indicator_list)

이동 평균을 계산했습니다.
RSI를 계산했습니다.


In [82]:
indicator_list_more = [HighPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

수행한 숫자:11395
가장 높은 점수:200


In [83]:
indicator_list_more = [LowPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

In [84]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score
0,2023-01-01 00:00:00,16583.5,16583.5,16608.0,16600.0,1.672499e+09,16600.000000,16600.000000,0.000000,0,0
10,2023-01-01 00:15:00,16591.5,16583.5,16608.0,16590.5,1.672500e+09,16595.250000,16595.250000,0.000000,0,0
27,2023-01-01 00:45:00,16590.5,16578.0,16590.5,16580.0,1.672502e+09,16590.166667,16590.166667,0.000000,0,0
47,2023-01-01 01:15:00,16583.0,16573.5,16588.5,16588.5,1.672503e+09,16589.750000,16589.750000,29.824561,0,0
66,2023-01-01 01:45:00,16587.5,16585.5,16596.5,16592.5,1.672505e+09,16590.300000,16590.300000,38.461538,0,0
...,...,...,...,...,...,...,...,...,...,...,...
174825,2023-05-08 11:00:00,28308.0,28207.0,28374.0,28362.0,1.683511e+09,28675.150000,28868.091667,30.264244,0,0
174840,2023-05-08 11:15:00,28362.0,28293.0,28374.0,28329.5,1.683512e+09,28642.475000,28858.633333,30.276745,0,0
174855,2023-05-08 11:30:00,28329.5,28260.0,28330.5,28330.5,1.683513e+09,28614.275000,28850.125000,29.519833,0,0
174870,2023-05-08 11:45:00,28330.5,28315.5,28395.0,28376.0,1.683514e+09,28590.850000,28842.741667,32.255457,0,0


In [88]:
process_instance = GetHighPoints(bitmex.data, 'high_score', 0.95)
high_points = bitmex.get_key_points(process_instance)
high_points

정형화된 기준값은 : 0.005


C:\Users\User\AppData\Local\Temp/ipykernel_6076/2166898150.py:263: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
C:\Users\User\AppData\Local\Temp/ipykernel_6076/2166898150.py:263: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df_calculated.loc[df_calculated['normalized_value_high'] >= threshold, 'origin_high_score'] = df_calculated[target_column]
C:\Users\User\AppData\Local\Temp/ipykernel_6076/216689815

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60,RSI,high_score,low_score,normalized_value_high,origin_high_score,high_for_graph
2329,2023-01-03 06:45:00,16714.0,16714.0,16789.5,16740.5,1.672696e+09,16712.150,16685.916667,58.436214,59.0,0,0.295,59,16789.5
3785,2023-01-04 12:00:00,16724.0,16723.0,16832.0,16828.0,1.672801e+09,16672.125,16665.083333,84.570312,1.0,0,0.005,1,16832.0
3800,2023-01-04 12:15:00,16828.0,16787.5,16837.5,16793.0,1.672802e+09,16679.500,16666.350000,73.534636,2.0,0,0.010,2,16837.5
3829,2023-01-04 12:45:00,16805.5,16805.5,16866.5,16857.0,1.672804e+09,16698.450,16670.541667,80.415430,1.0,0,0.005,1,16866.5
3844,2023-01-04 13:00:00,16857.0,16855.5,16904.0,16856.0,1.672805e+09,16709.450,16673.100000,79.940120,17.0,0,0.085,17,16904.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169682,2023-05-04 19:30:00,29180.5,29171.5,29400.0,29294.5,1.683196e+09,29119.550,29021.166667,66.041276,62.0,0,0.310,62,29400.0
170597,2023-05-05 11:00:00,29330.0,29250.0,29549.5,29279.0,1.683252e+09,28919.050,28914.033333,85.912409,55.0,0,0.275,55,29549.5
171431,2023-05-06 01:00:00,29380.0,29380.5,29680.0,29619.0,1.683302e+09,29195.625,29176.191667,80.599144,1.0,0,0.005,1,29680.0
171446,2023-05-06 01:15:00,29619.0,29540.0,29710.0,29545.5,1.683303e+09,29217.400,29186.700000,76.145553,33.0,0,0.165,33,29710.0


In [74]:
bitmex.data.loc[bitmex.data['high_score'].idxmax()]['high_score']

200

In [42]:
bitmex.data['high_score']

0         0
10        0
27        0
47        0
66        0
         ..
174825    0
174840    0
174855    0
174870    0
174885    0
Name: high_score, Length: 11595, dtype: int64